# Lab - First Request: The JSON Behind One Call

Three tasks on the one call every Claude feature starts with - read the way a developer reads it: a JSON request, a response you log, and a stop reason your code checks.

| Task | What you learn |
| --- | --- |
| 1 | The request is a JSON body; the key stays in the environment, not the code |
| 2 | Copy id, model, stop_reason and usage into one log line per request |
| 3 | A reply cut by max_tokens is an error in code - catch it and ask again |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
# No key in the code: the client reads ANTHROPIC_API_KEY from the environment,
# which load_dotenv() filled from the .env file.
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
QUESTION = ("A customer asks: can I return a jacket after 20 days? Our policy "
            "allows returns within 30 days. Answer the customer in two sentences.")

---

## Task 1 - The request is plain JSON

In [ ]:
# ============================================================
# TASK 1 - The request is plain JSON
# ============================================================
#
# WHAT TO DO
#   Write the request as one dictionary, send it with
#   client.messages.create(**REQUEST), and print it as JSON.
#
# WHY IT MATTERS
#   The SDK is a thin wrapper: what it sends is a JSON body to
#   the /v1/messages endpoint. Four things go in it - the model,
#   max_tokens (the most tokens Claude may write), the effort,
#   and the messages so far. The key is not one of them: it
#   travels in a header, read from the environment. A request
#   you can print as JSON is a request you can log, test and
#   replay.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The request itself is one method call".
#
# HOW TO DO IT
#   Three blanks:
#
#       MODEL      the model constant from the setup cell
#       300        the most tokens Claude may write
#       QUESTION   the customer's question, as the user message
# ============================================================

# TODO: replace each ... below with the value named beside it
REQUEST = {
    "model": ...,        # MODEL
    "max_tokens": ...,   # 300 - a ceiling, not a target
    "output_config": {"effort": "low"},
    "messages": [{"role": "user", "content": ...}],   # QUESTION
}
response = client.messages.create(**REQUEST)

print(json.dumps(REQUEST, indent=2))   # the body of the HTTPS request

check("request_json", request=REQUEST, response=response)

---

## Task 2 - Turn the response into a log record

In [ ]:
# ============================================================
# TASK 2 - Turn the response into a log record
# ============================================================
#
# WHAT TO DO
#   Copy the fields that matter out of the response into one
#   dictionary, and print it as a single JSON line.
#
# WHY IT MATTERS
#   The response is a Message object. content is a list of
#   blocks - for a plain answer, one text block. stop_reason
#   says why Claude stopped. usage counts the tokens sent and
#   written. An application logs these on every call: the id
#   to find the request again, stop_reason to spot cut
#   replies, usage to add up the bill.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now let's read the response".
#
# HOW TO DO IT
#   Three blanks:
#
#       response.stop_reason          why Claude stopped
#       response.usage.output_tokens  the tokens Claude wrote
#       response.content[0].text      the reply itself
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    record = {
        "id": response.id,
        "model": response.model,
        "stop_reason": ...,     # response.stop_reason
        "input_tokens": response.usage.input_tokens,
        "output_tokens": ...,   # response.usage.output_tokens
        "text": ...,            # response.content[0].text
    }
    print(json.dumps(record))
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("log_record", record=record, response=response)

---

## Task 3 - A cut reply is an error in code

In [ ]:
# ============================================================
# TASK 3 - A cut reply is an error in code
# ============================================================
#
# WHAT TO DO
#   Finish finish(): it returns the text only when stop_reason
#   is "end_turn", and raises IncompleteReply otherwise. Then
#   send the same request with max_tokens=20, catch the error,
#   and ask again with room to finish.
#
# WHY IT MATTERS
#   max_tokens is a ceiling: Claude stops the moment it reaches
#   it, even mid-sentence, and the API reports success. If
#   ShopAssist showed that text, the customer would see half a
#   sentence. Code that reads stop_reason first - and treats a
#   cut reply as an error - never passes the fragment on.
#
# WHERE TO SEE IT IN THE LECTURE
#   "It is tempting to think of max tokens as the length we
#   want".
#
# HOW TO DO IT
#   Three blanks:
#
#       "end_turn"   the only stop_reason that means complete
#       20           the ceiling that cuts the reply
#       300          room to finish, for the second try
# ============================================================


class IncompleteReply(Exception):   # provided
    """The reply stopped before Claude finished it."""


def ask(max_tokens):   # provided: the same request, another ceiling
    return client.messages.create(**{**REQUEST, "max_tokens": max_tokens})


try:
    # TODO: replace each ... below with the value named beside it
    def finish(response):
        if response.stop_reason != ...:   # "end_turn"
            raise IncompleteReply(response.stop_reason)
        return response.content[0].text
    
    short = ask(...)   # 20 - the ceiling that cuts the reply
    try:
        answer = finish(short)
    except IncompleteReply as cut:
        print("cut by", cut, "- asking again with more room")
        answer = finish(ask(...))   # 300
except IncompleteReply as exc:   # provided: the second try was refused too
    raise ValueError("finish() refused the second reply as well (stop_reason {}). It "
                     'should raise only when stop_reason is not "end_turn".'.format(exc)) from None

print("cut reply:", repr(short.content[0].text))
print("answer:   ", answer)

check("incomplete_reply", short=short, answer=answer, finish=finish)

---

## Done

One call, read like a developer: the request is a JSON body without the key, the response becomes a log line, and stop_reason decides in code whether the reply can be used.